In [ ]:
# %pip install pandas
# %pip install psycopg2
# %pip install sqlalchemy
# %pip install dotenv 
# %pip install pydantic

# %pip freeze > ../requirements.txt

#### Disclaimer:

- This notebook helped me developing the tool, I intend to keep it here to show a little bit my process of development.

- I prioritize running script everytime I have a testing subject and I always make sure to test as many scenarios as I can before merging/deploying anything. The Jupyter Notebook helps to run python in chucks while we can easily print values everytime to check I/O.

### Data Ingestion:
---

In [2]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from time import sleep


def get_secrets() -> dict:
  '''Abstraction of the security layer for retrieving any secrets.'''
  
  ## IMPROVEMENT: Can be replaced by standard Auth methods
  
  load_dotenv()
  database_url = os.getenv('DATABASE_URL')


  if not database_url:
    raise Exception("secrets not found in environment variables.")
  
  secrets = {
    'database_url': database_url,
    ## Can contain other secrets
  } 

  return secrets
  

class raw_tools:
  '''A class for data ingestion tools. Currently supports CSV files.'''
  def __init__(self):
    pass
   
  @staticmethod  
  def csv_reader(
    file_path,
    **kwargs
    ) -> pd.DataFrame:
    '''Reads a CSV file and returns a DataFrame. Accepts additional keyword arguments for pd.read_csv()'''
    try:
      data = pd.read_csv(file_path, **kwargs)
      print(f".csv file read successfully from {file_path}. Shape: {data.shape}\n")
      return data
    except Exception as e:
      raise Exception(f"Error during csv_reader: {e}")
  
  @staticmethod  
  def dataframe_schema_definer(
    dataframe: pd.DataFrame,
    data_types: dict
    ) -> pd.DataFrame:
    '''Defines the schema of a DataFrame by converting columns to specified data types.'''
    try:
      for column, dtype in data_types.items():
        dataframe[column] = dataframe[column].astype(dtype, errors='ignore')
      print(f"DataFrame schema defined successfully.\n")
      return dataframe
    except Exception as e:
      raise Exception(f"Error during dataframe_schema_definer: {e}")
  
  @staticmethod
  def raw_data_loader(
    data: pd.DataFrame, 
    table_name: str,
    schema_name: str, 
    connection_string: str, 
    if_exists: str = 'replace'
    ) -> int:
    '''Function that loads a pandas dataframe to postgres table'''
    try:
      engine = create_engine(connection_string)
      result = data.to_sql(name=table_name, schema=schema_name, con=engine, if_exists=if_exists, index=False)
      print(f"Data loaded successfully into {table_name} table. Rows affected: {result}\n")
      return result
    except Exception as e:
      raise Exception(f"Error during raw_data_loader: {e}")
  
  @staticmethod
  def raw_verification(
    connection_string: str,
    table_name: str,
    schema_name: str,
    expected_rows: int,
    ) -> bool:
    '''Function that verifies the data loaded to postgres by querying the database metadata'''
    try:
      
      engine = create_engine(connection_string)
      
      ## Query row count from pg_class (IMPROVEMENT: Can be replaced by another "INFORMATION_SCHEMA" query for other SQL, like MySQL...)
      row_count_query = f"""
      SELECT 
        n_live_tup as row_count
      FROM pg_stat_user_tables
      WHERE 
        schemaname = '{schema_name}'
        AND relname = '{table_name}'
      """
      
      result = pd.read_sql_query(row_count_query, engine)
      
      if result.empty:
          raise Exception(f"Table {schema_name}.{table_name} not found.")
      
      actual_rows = result['row_count'].values[0]
      
      if actual_rows != expected_rows:
          raise Exception(
              f"Table length verification failed: Expected {expected_rows} rows, "
              f"but found {actual_rows} rows in {schema_name}.{table_name} table."
          )
      
      print(f"raw_verification successful for {schema_name}.{table_name} table. Row count: {actual_rows}, Expected: {expected_rows}\n")
      return True

    except Exception as e:
      print(f"raw_verification failed for {schema_name}.{table_name} table. Error: {e}")
      return False


class raw_data(raw_tools):
  '''
  Class for processing and storing raw data
  
  Properties:
    - `table_name`: str - name of the table to load data into
    - `schema_name`: str - name of the schema to load data into
  '''
  def __init__(
    self,
    table_name: str,
    schema_name: str,
    ) -> None:
    self.table_name: str = table_name
    self.schema_name: str = schema_name
    
  def ingest_data_structured(
    self,
    file_path: str,
    connection_string: str,
    table_schema_mapping: dict,
    ):
    '''Ingest data from CSV, stores locally (Runtime)'''

    ingestion_step = 0

    try:
      
      print(f"\n-- Processing {self.schema_name}.{self.table_name} table ingestion and verification. --\n")
      
      ## Create ingested dataframe
      self.dataframe = self.csv_reader(file_path) ## step 0
      ingestion_step += 1
      self.typed_dataframe = self.dataframe_schema_definer(self.dataframe, table_schema_mapping) ## step 1
      ingestion_step += 1
      
      ## Let's say a problem occours here, you can start the verification from here. E.g. Airflow shows you the task that failed
      ## raise(Exception("aaaaaaa")) 
      
      ## Load data to Postgres and verify
      self.row_count_expectation = self.raw_data_loader(
        data=self.typed_dataframe,
        table_name=self.table_name,
        schema_name=self.schema_name,
        connection_string=connection_string
        ) ## step 2
      ingestion_step += 1
      
      sleep(10) ## Avoid potential noise from database between transactions
      self.verification_flag = self.raw_verification(
        connection_string=connection_string,
        table_name=self.table_name,
        schema_name=self.schema_name,
        expected_rows=self.row_count_expectation
      ) ## step 3
        
      print(f"Data ingestion completed for {self.schema_name}.{self.table_name} table. Verification flag: {self.verification_flag}\n")
      
    except Exception as e:
      raise Exception(f"Error during ingest_data_structured: {e}. Stopped at ingestion step: {ingestion_step}")



## --------------- __main__ ---------------: 


connection_string = get_secrets()['database_url']

## unit.csv data:

unit = raw_data(
  table_name='unit',
  schema_name='monument_raw'
  )

unit_data_types = {
  'facilityName': 'string',
  'unitNumber': 'int64',
  'unitSize': 'string',
  'unitType': 'string',
  }

unit.ingest_data_structured(
  file_path='../data/unit.csv',
  connection_string=connection_string,
  table_schema_mapping=unit_data_types,
  ) 

## rentRoll.csv data:

rentRoll = raw_data(
  table_name='rent_roll',
  schema_name='monument_raw'
  )

rentRoll_data_types = {
  'facilityName': 'string',
  'unitNumber': 'int64',
  'firstName': 'string',
  'lastName': 'string',
  'phone': 'string',
  'email': 'string',
  'rentStartDate': 'string',
  'rentEndDate': 'string',
  'monthlyRent': 'float32',
  'currentRentOwed': 'float32',
  }

rentRoll.ingest_data_structured(
  file_path='../data/rentRoll.csv',
  connection_string=connection_string,
  table_schema_mapping=rentRoll_data_types,
  )


-- Processing monument_raw.unit table ingestion and verification. --

.csv file read successfully from ../data/unit.csv. Shape: (9, 4)

DataFrame schema defined successfully.

Data loaded successfully into unit table. Rows affected: 9

raw_verification successful for monument_raw.unit table. Row count: 9, Expected: 9

Data ingestion completed for monument_raw.unit table. Verification flag: True


-- Processing monument_raw.rent_roll table ingestion and verification. --

.csv file read successfully from ../data/rentRoll.csv. Shape: (4, 10)

DataFrame schema defined successfully.

Data loaded successfully into rent_roll table. Rows affected: 4

raw_verification successful for monument_raw.rent_roll table. Row count: 4, Expected: 4

Data ingestion completed for monument_raw.rent_roll table. Verification flag: True



In [ ]:
# testi = unit.dataframe.copy()
# # testi = testi.dropna(axis=0, how='any')

# # If N/A, then ignore errors and set to NaN (pandas default behavior)
# testi['unitNumber'] = testi['unitNumber'].astype('int64', errors='ignore')
# testi['facilityName'] = testi['facilityName'].astype('string', errors='ignore')
# testi

### Data Transformation 1:

---

In [ ]:
## Apply the transformation scripts on each table (5 tables - five functions)
## Column transformations belongs to each column, for simplicity (they wouldn't have complex dependencies then)
## >> Validates the data (e.g., ensures required fields are present; no orphaned foreign keys).

# - While you may not implement all things things due to time constraints be prepared to discuss
# Scaling the job for large data sets
# Handling rollbacks / corrections due to errors
# Being able to append updates to the data to the database
# Data security

In [7]:
import pandas as pd
from dateutil import parser
from sqlalchemy import create_engine
from time import sleep


def date_parser(date_str):
  '''Simple dateparser'''
  try:
    return parser.parse(date_str)
  except (parser.ParserError, TypeError):
    return None

## monument.facility table:
def build_facility_table(
  raw_unit: pd.DataFrame
  ) -> pd.DataFrame:
  '''Builds the facility table.'''
  
  try:
    ## Logic to build the table:
    facility_df = raw_unit.copy()
    facility_df = raw_unit.groupby('facilityName').agg({'facilityName': 'first'}).reset_index(drop=True) ## doesn't include Nulls by default
    facility_df.rename(columns={'facilityName': 'name'}, inplace=True)
    facility_df['facilityid'] = facility_df.index + 1 ## IMPROVEMENT: pegar o tamanho da tabela já no banco e somar +1
    facility_df = facility_df[['facilityid', 'name']]
  except Exception as e:
    raise Exception(f"Error during build_facility_table() while applying logic: {e}")
  
  try:
    ## Validates data schema:
    facility_df['name'] = facility_df['name'].astype(str) 
    if facility_df['name'].str.len().max() > 100:
      exceptions = facility_df[facility_df['name'].str.len() > 100]['name'].tolist()
      raise Exception(f"Facility Name has more than 100 characters: {exceptions}")
  except Exception as e:
    raise Exception(f"Error during build_facility_table() while validating schema: {e}")
  
  return facility_df


## monument.unit table:
def build_unit_table(
  facility_df: pd.DataFrame,
  raw_unit: pd.DataFrame,
  raw_rentRoll: pd.DataFrame
  ):
  '''Builds the unit table.'''
  
  try:
    ## Logic to build the table:
    unit_silver_df = raw_unit.copy()
    unit_silver_df = unit_silver_df.dropna(subset=['unitNumber', 'facilityName'])
    unit_silver_df['unitSize'] = unit_silver_df['unitSize']\
      .str.replace('  ', ' ')\
      .str.replace('(', '')\
      .str.replace(')', '')\
      .str.upper()\
      .str.strip()\
      .str.split(' ')
    unit_silver_df['unitSize'] = unit_silver_df['unitSize'].apply(
      lambda sizearray: [dimension.split('X') for dimension in sizearray] if sizearray is not None else None
    )
    unit_silver_df['unitwidth'] = unit_silver_df['unitSize'].apply(
      lambda sizearray: sizearray[0][sizearray[1].index('W')] if sizearray is not None else None
    )
    unit_silver_df['unitlength'] = unit_silver_df['unitSize'].apply(
      lambda sizearray: sizearray[0][sizearray[1].index('L')] if sizearray is not None else None
    )
    unit_silver_df['unitheight'] = unit_silver_df['unitSize'].apply(
      lambda sizearray: sizearray[0][sizearray[1].index('H')] if sizearray is not None else None
    )
    unit_silver_df.drop(columns=['unitSize'], inplace=True)
    unit_silver_df['unitNumber'] = unit_silver_df['unitNumber'].astype(int)
    unit_silver_df = unit_silver_df.merge(
      raw_rentRoll[['facilityName', 'unitNumber', 'monthlyRent']].drop_duplicates(),
      on=['facilityName', 'unitNumber'],
      how='left'
    )
    unit_silver_df = unit_silver_df.merge(
      facility_df[['facilityid', 'name']],
      left_on='facilityName',
      right_on='name',
      how='left'
    )
    unit_silver_df = unit_silver_df.sort_values(by=['facilityName','unitNumber']).reset_index(drop=True)
    unit_silver_df = unit_silver_df.drop_duplicates().reset_index(drop=True)
    unit_silver_df['unitid'] = unit_silver_df.index + 1 ## IMPROVEMENT: pegar o tamanho da tabela já no banco e somar +1
    unit_silver_df.drop(columns=['facilityName','name'], inplace=True)
    unit_silver_df.rename(columns={
      'unitNumber': 'number',
      'unitType': 'unittype',
      'monthlyRent': 'monthlyrent'
    }, inplace=True)
    unit_silver_df = unit_silver_df[['unitid','facilityid', 'number', 'unitwidth', 'unitlength', 'unitheight', 'unittype', 'monthlyrent']]
    return unit_silver_df
  
  except Exception as e:
    raise Exception(f"Error during build_unit_table() while applying logic: {e}")
  
  
## monument.tenant table:
def build_tenant_table(
  raw_rentRoll: pd.DataFrame,
):
  '''Builds the tenant table.'''
  
  try:
    tenant_df = raw_rentRoll.copy()
    ## take out any special charaacter, doesn't cover numbers with less then 9 or more than 10 digits yet:
    tenant_df['phone'] = tenant_df['phone'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' ', '')
    tenant_df.rename(columns={
      'firstName': 'firstname',
      'lastName': 'lastname'
    }, inplace=True)
    tenant_df['firstname'] = tenant_df['firstname'].str.strip()
    tenant_df['lastname'] = tenant_df['lastname'].str.strip()
    tenant_df = tenant_df[['firstname', 'lastname', 'phone', 'email']]
    tenant_df= tenant_df.drop_duplicates().reset_index(drop=True)
    tenant_df['tenantid'] = tenant_df.index + 1 ## IMPROVEMENT: pegar o tamanho da tabela já no banco e somar +1
    
    return tenant_df

  except Exception as e:
    raise Exception(f"Error during build_tenant_table() while applying logic: {e}")
  
  

## monument.rentalContract table:
def build_rentalContract_table(
  raw_rentRoll: pd.DataFrame,
  facility_silver_df: pd.DataFrame,
  unit_silver_df: pd.DataFrame,
  tenant_silver_df: pd.DataFrame
):
  '''Builds the rental contract table.'''
  
  try:
    
    unit_silver_df = unit_silver_df.copy()
    unit_silver_df = unit_silver_df.merge(
      facility_silver_df[['facilityid', 'name']],
      left_on='facilityid',
      right_on='facilityid',
      how='left'
    )
    unit_silver_df.rename(columns={'name': 'facilityname'}, inplace=True)
    
    rental_contract_df = raw_rentRoll.copy()
    rental_contract_df = rental_contract_df.dropna(subset=['rentStartDate'])
    
    rental_contract_df['firstName'] = rental_contract_df['firstName'].str.strip()
    rental_contract_df['lastName'] = rental_contract_df['lastName'].str.strip()
    
    ## join with tenant for tenantId
    rental_contract_df = rental_contract_df.merge(
      tenant_silver_df[['firstname', 'lastname', 'email', 'tenantid']],
      left_on=['firstName', 'lastName', 'email'],
      right_on=['firstname', 'lastname', 'email'],
      how='left'
    )
    
    ## join with unit for unitId
    rental_contract_df = rental_contract_df.merge(
      unit_silver_df[['facilityid', 'facilityname', 'number', 'unitid']],
      left_on=['facilityName', 'unitNumber'],
      right_on=['facilityname', 'number'],
      how='left'
    )
        
    rental_contract_df['rentalContractid'] = rental_contract_df.index + 1 ## IMPROVEMENT: pegar o tamanho da tabela já no banco e somar +1
    rental_contract_df.rename(columns={
      'rentStartDate': 'startdate',
      'rentEndDate': 'enddate',
      'currentRentOwed': 'currentamountowed',
      'rentalContractid': 'rentalcontractid'
    }, inplace=True)
    
    rental_contract_df = rental_contract_df[['rentalcontractid','unitid','tenantid','startdate', 'enddate', 'currentamountowed']]
    rental_contract_df['startdate'] = rental_contract_df['startdate'].apply(
      lambda x: date_parser(x)
    )
    rental_contract_df['enddate'] = rental_contract_df['startdate'].apply(
      lambda x: date_parser(x)
    )
    
    return rental_contract_df

  except Exception as e:
    raise Exception(f"Error during build_rentalContract_table() while applying logic: {e}")
  

## monument.rentalInvoice table:
def build_rentalInvoice_table(
  raw_rentRoll: pd.DataFrame,
  rental_contract_silver_df: pd.DataFrame,
  unit_silver_df: pd.DataFrame,
):
  '''Builds the rental invoice table.'''
  try:
    
    ## Preparing data for collecting the rentalContractid
    rental_contract_silver_df = rental_contract_silver_df.copy()
    rental_contract_silver_df = rental_contract_silver_df.merge(
      unit_silver_df[['monthlyrent', 'unitid', 'number']],
      left_on='unitid',
      right_on='unitid',
      how='left'
    )
  
    rental_invoice_df = raw_rentRoll.copy()
    rental_invoice_df['invoiceduedate'] = rental_invoice_df['rentStartDate'].apply(
      lambda x: date_parser(x)
    )
    rental_invoice_df['invoiceduedate'] = rental_invoice_df['invoiceduedate'] + pd.offsets.MonthBegin(1) ## first day of the next month
    
    # As I did join by unitid before, I'll be having "unique unit numbers" at this step
    rental_invoice_df = rental_invoice_df.merge(
      rental_contract_silver_df[['rentalcontractid', 'number', 'monthlyrent']],
      left_on='unitNumber',
      right_on='number',
      how='left'
    )
    
    rental_invoice_df['invoiceid'] = rental_invoice_df.index + 1 ## IMPROVEMENT: pegar o tamanho da tabela já no banco e somar +1
    
    rental_invoice_df.rename(columns={
      'monthlyrent': 'invoiceamount',
      'currentRentOwed': 'invoicebalance',
      'rentalContractid': 'rentalcontractid',
    }, inplace=True)
    
    # rental_invoice_df.dropna(subset=['rentalcontractid'], inplace=True)
    
    rental_invoice_df = rental_invoice_df[['invoiceid', 'rentalcontractid', 'invoiceduedate', 'invoiceamount', 'invoicebalance']]
    return rental_invoice_df
  
  except Exception as e:
    raise Exception(f"Error during build_rentalInvoice_table() while applying logic: {e}")
  

###### Pipeline functions:

def silver_data_load(
  data: pd.DataFrame, 
  table_name: str,
  schema_name: str, 
  connection_string: str, 
  if_exists: str = 'append'
  ):
  '''Load data into silver layer'''
  try:
    engine = create_engine(connection_string)
    rows_affected = data.to_sql(name=table_name, schema=schema_name, con=engine, if_exists=if_exists, index=False)
    print(f"Data loaded successfully into {table_name} table. Rows affected: {rows_affected}\n")
      
    ## Fazer validação do insert
   
    return rows_affected
  except Exception as e:
    raise Exception(f" >>>> Error during silver_data_load(): {e}")
    ## to_sql method will present any error before actually insert any data (so, rollback is 'automatic')

def retreive_raw_data():
  '''Retrieve data from raw layer'''
  
  connection_string = get_secrets()['database_url']
  engine = create_engine(connection_string)
  
  ## Option, can change the methods here to pull directly from runtime, instead of pulling from SQL Database
  raw_unit_df = pd.read_sql_table(table_name='unit', schema='monument_raw', con=engine)
  raw_rentRoll_df = pd.read_sql_table(table_name='rentRoll', schema='monument_raw', con=engine)
  
  if raw_unit_df.empty or raw_rentRoll_df.empty:
    raise Exception("One of the raw tables is empty. Please check the raw layer.")
  
  return raw_unit_df, raw_rentRoll_df


def silver_verification(
    connection_string: str,
    table_name: str,
    schema_name: str,
    expected_rows: int,
    ) -> bool:
    '''Function that verifies the data loaded to postgres by querying the database metadata'''
    
    ## a bit messy because I didn't call them a good name in time (lack of time)
    success = raw_tools.raw_verification(
      connection_string=connection_string,
      table_name=table_name,
      schema_name=schema_name,
      expected_rows=expected_rows
    )
    if not success:
      raise Exception(f"Verification failed for {schema_name}.{table_name} table. Expected rows: {expected_rows}")
    
    return True
      

# def silver_pipeline():
#   '''Function that runs the silver pipeline.'''
#   try:
    
    # connection_string = get_secrets()['database_url']
    
raw_unit_df, raw_rentRoll_df = retreive_raw_data()
## Creating tables (as pandas dataframes)
## If a problem occours in this step, no data will be uploaded.
facility_silver_df = build_facility_table(raw_unit_df)
unit_silver_df = build_unit_table(facility_silver_df, raw_unit_df, raw_rentRoll_df)
tenant_silver_df = build_tenant_table(raw_rentRoll_df)
rental_contract_silver_df = build_rentalContract_table(raw_rentRoll_df, facility_silver_df, unit_silver_df, tenant_silver_df)
rental_invoice_silver_df = build_rentalInvoice_table(raw_rentRoll_df, rental_contract_silver_df, unit_silver_df)

    ## Loading tables to silver layer:
    ## If a problem occours in this block, i.e. on the "tenant_silver_df", it will not load problematic data, 
    ## the previous data will be correct, and the next tables wil be blank yet.
    ## I've separated individually the loads since we could be calling these "Lambda" functions from a queue, and resume the queue from where it failed.
    # facility_silver_exp_rows = silver_data_load(data=facility_silver_df,table_name='facility',schema_name='monument', connection_string=connection_string)
    # unit_silver_exp_rows = silver_data_load(data=unit_silver_df,table_name='unit',schema_name='monument', connection_string=connection_string)
    # tenant_silver_exp_rows = silver_data_load(data=tenant_silver_df,table_name='tenant',schema_name='monument', connection_string=connection_string)
    # # raise(Exception("ISSUE HEREEE")) ## Feel free to test here
    # rental_contract_silver_exp_rows = silver_data_load(data=rental_contract_silver_df, table_name='rentalContract', schema_name='monument', connection_string=connection_string)
    # rental_invoice_silver_exp_rows = silver_data_load(data=rental_invoice_silver_df,  table_name='rentalInvoice',  schema_name='monument', connection_string=connection_string)

    # print("###### Now validating the data for the monument schema ######")

    # ### Validating data load:
    # sleep(10) # to enable time for postgreSQL to refresh
    # silver_verification(connection_string=connection_string, table_name='facility', schema_name='monument', expected_rows=facility_silver_exp_rows)
    # silver_verification(connection_string=connection_string, table_name='unit', schema_name='monument', expected_rows=unit_silver_exp_rows)
    # silver_verification(connection_string=connection_string, table_name='tenant', schema_name='monument', expected_rows=tenant_silver_exp_rows)
    # silver_verification(connection_string=connection_string, table_name='rentalContract', schema_name='monument', expected_rows=rental_contract_silver_exp_rows)
    # silver_verification(connection_string=connection_string, table_name='rentalInvoice', schema_name='monument', expected_rows=rental_invoice_silver_exp_rows)

  #   return True

  # except Exception as e:
  #   raise Exception(f"Error during silver_pipeline(): {e}")
  

## ---------------- __main__ ----------------:
rental_contract_silver_df




# raw_rentRoll_df_test = raw_rentRoll_df.copy()
# raw_rentRoll_df_test = pd.concat([raw_rentRoll_df_test, pd.DataFrame({
#   'facilityName': ['any'],
#   'unitNumber': [9999],
#   'firstName': ['any'],
#   'lastName': ['any'],
#   'phone': ['+1 (555) 555-5555'],
#   'email': ['any'],
#   'rentStartDate': ['2024-01-01'],
#   'rentEndDate': ['2024-12-31'],
#   'monthlyRent': [999.99],
#   'currentRentOwed': [999.99],
# })], ignore_index=True)


# raw_unit_df_test = raw_unit_df.copy()
# raw_unit_df_test = pd.concat([raw_unit_df_test, pd.DataFrame({
#   'facilityName': ['any'],
#   'unitNumber': ['any'],
#   'unitSize': [None],
#   'unitType': ['any']
# })], ignore_index=True)

,rentalcontractid,unitid,tenantid,startdate,enddate,currentamountowed
0,1,2,1,2026-04-01,None,2300.0
1,2,1,1,2026-03-01,None,0.0
2,3,6,2,2026-03-20,None,0.0
3,4,7,3,2026-01-01,None,5000.0


In [4]:
tenant_silver_df

,firstname,lastname,phone,email,tenantid
0,Joao,Santos,2015550123,joao.santos@monument.io,1
1,Matthew,Hatch,2015550124,matthew.hatch@monument.io,2
2,Augusto,Giani,2015559999,augusto.giani@monument.io,3


In [10]:
tenant_silver_df = tenant_silver_df[['firstname', 'lastname', 'phone', 'email']]
tenant_silver_df = tenant_silver_df.drop_duplicates(inplace=True).reset_index()
tenant_silver_df

AttributeError: 'NoneType' object has no attribute 'reset_index'

### Testing the script's modules:

---

From this point, I changed the modules already for script runtime (not notebook)

In [11]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from lib.raw_tools import raw_data, get_secrets
# from lib.silver_tools import silver_pipeline

def raw_pipeline():
  '''Runs the initial ingestion piece'''
  try:
    connection_string = get_secrets()['database_url']

    ## unit.csv data:

    unit = raw_data(
      table_name='unit',
      schema_name='monument_raw'
      )

    unit_data_types = {
      'facilityName': 'string',
      'unitNumber': 'int64',
      'unitSize': 'string',
      'unitType': 'string',
      }

    unit.ingest_data_structured(
      file_path='../data/unit.csv',
      connection_string=connection_string,
      table_schema_mapping=unit_data_types,
      ) 

    ## rentRoll.csv data:

    rentRoll = raw_data(
      table_name='rentRoll',
      schema_name='monument_raw'
      )

    rentRoll_data_types = {
      'facilityName': 'string',
      'unitNumber': 'int64',
      'firstName': 'string',
      'lastName': 'string',
      'phone': 'string',
      'email': 'string',
      'rentStartDate': 'string',
      'rentEndDate': 'string',
      'monthlyRent': 'float32',
      'currentRentOwed': 'float32',
      }

    rentRoll.ingest_data_structured(
      file_path='../data/rentRoll.csv',
      connection_string=connection_string,
      table_schema_mapping=rentRoll_data_types,
      )
  
    return True
  except Exception as e:
    print(f"Error during raw_pipeline() execution: {e}")
    return False

def silver_main_pipeline():
  '''Creates and populates the defined tables'''
  try:
    flag = silver_pipeline()    
    if flag:
      print("Ingestion process completed successfully.")
    return flag
  except Exception as e:
    print(f"Error during silver_main_pipeline() execution: {e}")
    return False

if __name__ == "__main__":
  
  ## Main run:    
  step1_flag = raw_pipeline()
  
  print("--------------------------------------------------------\n")
  
  if step1_flag:
    silver_main_pipeline()


-- Processing monument_raw.unit table ingestion and verification. --

.csv file read successfully from ../data/unit.csv. Shape: (5, 4)

DataFrame schema defined successfully.

Data loaded successfully into unit table. Rows affected: 5

raw_verification successful for monument_raw.unit table. Row count: 5, Expected: 5

Data ingestion completed for monument_raw.unit table. Verification flag: True


-- Processing monument_raw.rentRoll table ingestion and verification. --

.csv file read successfully from ../data/rentRoll.csv. Shape: (2, 10)

DataFrame schema defined successfully.

Data loaded successfully into rentRoll table. Rows affected: 2

raw_verification successful for monument_raw.rentRoll table. Row count: 2, Expected: 2

Data ingestion completed for monument_raw.rentRoll table. Verification flag: True

--------------------------------------------------------

Data loaded successfully into facility table. Rows affected: 2

Data loaded successfully into unit table. Rows affected: 